# Building a Knowledge Graph from Unstructured Text (Step by Step)

This notebook walks through every stage of the pipeline in `src/` one
piece at a time, so you can inspect intermediate outputs - perfect for
a tutorial recording. For the full production run, use `python main.py`
instead (see the README).

**Pipeline stages:**
1. Data acquisition (BBC News dataset)
2. Text preprocessing & sentence segmentation
3. Named Entity Recognition + coreference resolution
4. Relation extraction (SVO triples)
5. Knowledge graph construction
6. Interactive visualization


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from src.utils import load_config, setup_logging
from src.data_loader import get_dataset
from src.preprocessing import preprocess_articles
from src.ner_extraction import extract_entities
from src.relation_extraction import extract_relations
from src.graph_builder import build_graph, export_graph, compute_graph_stats
from src.visualize import render_html

config = load_config('config.yaml')
# Use a small sample for a fast, interactive walkthrough
config['data']['sample_size'] = 40
logger = setup_logging(config)


## Stage 1 - Data Acquisition
Downloads (or reuses a cached copy of) the BBC News dataset - 2,225 real news articles.

In [ ]:
articles = get_dataset(config)
print(f"Loaded {len(articles)} articles")
articles.head()


## Stage 2 - Preprocessing
Cleans text and splits every article into sentences (the unit we run NLP over).

In [ ]:
sentences = preprocess_articles(articles, config)
print(f"{len(sentences)} sentences from {len(articles)} articles")
sentences.head(10)


## Stage 3 - Named Entity Recognition & Coreference

Runs spaCy's NER model over every sentence, keeps entities of the
configured types (PERSON, ORG, GPE, ...), and resolves simple pronouns
("it", "he", "they"...) back to the nearest matching entity.

In [ ]:
mentions = extract_entities(sentences, config)
print(f"{len(mentions)} entity mentions, {mentions['canonical'].nunique()} unique entities")
print(f"{mentions['resolved_from_pronoun'].sum()} mentions resolved via pronoun coreference")
mentions.head(15)


In [ ]:
# Visualize entity type distribution
import matplotlib.pyplot as plt
mentions['label'].value_counts().plot(kind='bar', figsize=(8,4), title='Entity type distribution')
plt.tight_layout()
plt.show()


## Stage 4 - Relation Extraction

Walks spaCy's dependency parse tree for every sentence to pull out
(subject, relation, object) triples. This is fully rule-based and
inspectable - no black-box model.

In [ ]:
triples = extract_relations(sentences, config)
print(f"{len(triples)} raw triples extracted")
triples[['subject', 'relation', 'object']].head(20)


### Try it on your own sentence

Run the extractor on any sentence you type to see exactly what the
dependency parser is doing.

In [ ]:
import spacy
from src.relation_extraction import _extract_triples_from_sentence

nlp = spacy.load(config['nlp']['spacy_model'])

test_sentence = "Microsoft agreed to acquire Activision Blizzard for $69 billion in 2022."
doc = nlp(test_sentence)

print("Dependency parse:")
for token in doc:
    print(f"  {token.text:15} {token.dep_:12} head={token.head.text}")

print("\nExtracted triples:")
for s, r, o in _extract_triples_from_sentence(doc):
    print(f"  ({s}) -[{r}]-> ({o})")


## Stage 5 - Knowledge Graph Construction

Fuses entities + triples into one deduplicated graph. Repeated triples
across the corpus become a single edge with a `count` weight.

In [ ]:
graph = build_graph(mentions, triples, config)
stats = compute_graph_stats(graph)
print(f"Nodes: {stats['num_nodes']}  Edges: {stats['num_edges']}")
print(f"Connected components: {stats['num_connected_components']}")
print("\nTop entities by degree centrality:")
for e in stats['top_entities_by_degree_centrality'][:10]:
    print(f"  {e['entity']:35} ({e['label']:8}) score={e['score']}")


In [ ]:
export_graph(graph, config)  # writes GraphML/GEXF/CSV/stats.json to ../data/output/


## Stage 6 - Interactive Visualization

Renders a self-contained HTML file you can open in any browser -
drag nodes, zoom, hover for details. Great for screen-recording.

In [ ]:
html_path = render_html(graph, config)
print(f"Open this in your browser: {html_path}")


### Preview the graph inline (optional)

If you're running this notebook locally (not on a headless server),
this will embed the interactive graph directly below.

In [ ]:
from IPython.display import IFrame
IFrame(src=html_path, width='100%', height=800)


## Querying the graph directly with NetworkX

A few examples of questions you can now answer programmatically now
that unstructured text has become a structured graph.

In [ ]:
import networkx as nx

# Who/what is directly connected to a given entity?
entity = stats['top_entities_by_degree_centrality'][0]['entity']
print(f"Neighbors of '{entity}':")
for neighbor in graph.successors(entity):
    edge = graph.get_edge_data(entity, neighbor)
    print(f"  {entity} -[{edge['relation']}]-> {neighbor}  (seen {edge['count']}x)")
for neighbor in graph.predecessors(entity):
    edge = graph.get_edge_data(neighbor, entity)
    print(f"  {neighbor} -[{edge['relation']}]-> {entity}  (seen {edge['count']}x)")


In [ ]:
# Shortest path between two entities (if connected)
nodes = list(graph.nodes())
if len(nodes) >= 2:
    a, b = nodes[0], nodes[len(nodes)//2]
    try:
        path = nx.shortest_path(graph.to_undirected(), a, b)
        print(f"Path from '{a}' to '{b}':")
        print(" -> ".join(path))
    except nx.NetworkXNoPath:
        print(f"No path found between '{a}' and '{b}' (they're in different components).")


## Next steps

- Swap `sample_size` / `categories` in `config.yaml` to run over the
  full corpus.
- Try `--backend neo4j` (see README) to load this graph into Neo4j and
  query it with Cypher.
- Point `data.source_url` at your own corpus (reports, transcripts,
  Wikipedia dumps) - only a `text` column is required.
